# Fuerza de Marea

In [ ]:
import rebound
import reboundx
import numpy as np 
import matplotlib.pyplot as plt

In [ ]:
sim = rebound.Simulation()
sim.integrator = "ias15"
sim.collision = "direct"
sim.collision_resolve = "merge"
sim.units = ("AU", "yr", "Msun")

In [ ]:
M0 = 0.8645388227818771
R0 = 0.3833838293200158

sim.add(m=M0, r=R0)
sim.add(m=1e-3, a=2.0, e=0.5, r=0.0000477895)
sim.move_to_com()

In [ ]:
rebx = reboundx.Extras(sim)
tides = rebx.load_force("tides_constant_time_lag")
rebx.add_force(tides)
sim.particles[0].params["tctl_k2"] = 0.03
sim.particles[0].params["tctl_tau"] = 0.04
sim.particles[0].params["Omega"] = [0, 0, 0]

In [ ]:
sim.integrate(0.0)
sim.widget(size=(950,950))

In [ ]:
a1 = []
e1 = []
t1 = []
a2 = []
e2 = []
t2 = []
E0 = sim.energy()
Lx0, Ly0, Lz0 = sim.angular_momentum()
L0 = np.sqrt(Lx0**2 + Ly0**2 + Lz0**2)

err_ener=[]
err_ang=[]
tim = []

tmax = 4.e6
Num = 100000
times = np.linspace(0, tmax, Num)

for i, time in enumerate(times):
    sim.integrate(time)
    
    if sim.N > 1:
        a1.append(sim.particles[1].a)
        e1.append(sim.particles[1].e)
        t1.append(times[i])
    if sim.N > 2:
        a2.append(sim.particles[2].a)
        e2.append(sim.particles[2].e)
        t2.append(times[i])

    tim.append(sim.t)
    E_actual = sim.energy()
    err_ener.append(abs((E_actual - E0) / E0))

    Lx, Ly, Lz = sim.angular_momentum()
    L_actual = np.sqrt(Lx**2 + Ly**2 + Lz**2)
    err_ang.append(abs((L_actual - L0) / L0))

In [ ]:
fig = plt.figure(figsize=(15,5))

ax = plt.subplot(121)
ax.set_xlabel("time ([yr]")
ax.set_ylabel("semi-major axis [au]")
plt.plot(t1, a1, color="blue", label="planet 1")
ax.legend(loc="best")

ax = plt.subplot(122)
ax.set_xlabel("time [yr]")
ax.set_ylabel("eccentrecity")
plt.plot(t1, e1, color="blue", label="planet 1")
ax.legend(loc="best")

In [ ]:
fig = plt.figure(figsize=(15,5))

ax = plt.subplot(121)
ax.set_xlabel("time ([yr]")
ax.set_ylabel("err_ener")
plt.plot(tim, err_ener, color="red")

ax = plt.subplot(122)
ax.set_xlabel("time [yr]")
ax.set_ylabel("err_ang")
plt.plot(tim, err_ang, color="red")